# 📘 Notebook 5: Convexity, Saddle Points & Local vs Global Minima

**Week 2 — Calculus, Optimization & Probability Theory**

**Difficulty:** ⭐⭐⭐ (Intermediate)

---

## 🏠 Why Does This Matter?

You train your house price model for 1000 steps. Gradient descent converges — the loss stops decreasing.  
**But has it found the BEST possible weights, or just a "good enough" local solution?**

The answer depends on the shape of the loss surface:
- **Convex loss** (like MSE in linear regression): only ONE minimum — gradient descent always finds it
- **Non-convex loss** (neural networks): many local minima, saddle points — gradient descent finds *a* solution, which may not be optimal

This notebook explains how to recognize these situations and what to do about them.

---

## 📚 Prerequisites
- Notebooks 1–4 (derivatives, chain rule, gradient descent)

---

## Part 1: Convex Functions — The Easy Case

### Plain English First

A function is **convex** if its shape looks like a bowl:  
- No matter where you start, the bowl has exactly ONE lowest point
- Any path you draw inside the bowl stays inside the bowl
- If you draw a straight line between any two points on the bowl, the line lies **above** (or on) the bowl

**Why we love convex functions in ML:**  
Gradient descent on a convex function is **guaranteed to find the global minimum** — you can't get stuck!

**The chord test:** Draw a line between two points on the function.  
If the line is always **above** the function, it's convex.

$$f(\lambda x + (1-\lambda)y) \leq \lambda f(x) + (1-\lambda) f(y) \quad \text{for all } \lambda \in [0,1]$$

Plain English: "The midpoint of the function is below the midpoint of the chord between two points."

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.signal import argrelmin

# ─────────────────────────────────────────────────────────────────
# Visualize convex vs non-convex loss functions
# ─────────────────────────────────────────────────────────────────

w = np.linspace(-3, 3, 400)

# Convex: MSE loss as a function of a single weight w
# L_convex(w) = (w - 1)²  → minimum at w=1
L_convex    = (w - 1)**2

# Non-convex: what a neural network loss curve might look like
# This has multiple local minima (traps for gradient descent)
L_nonconvex = 0.5*w**4 - 2*w**2 + 0.3*w + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── Left: Convex ───────────────────────────────────────────────
axes[0].plot(w, L_convex, 'steelblue', linewidth=3, label='MSE Loss: L(w) = (w−1)²')

# Draw the chord between two points — it must be above the function
x1, x2 = -2.0, 2.5
y1, y2  = (x1-1)**2, (x2-1)**2
axes[0].plot([x1, x2], [y1, y2], 'red', linewidth=2.5, linestyle='--',
             label='Chord between two points')
axes[0].fill_between(w[(w>=x1)&(w<=x2)],
                     L_convex[(w>=x1)&(w<=x2)],
                     y1 + (y2-y1)/(x2-x1)*(w[(w>=x1)&(w<=x2)]-x1),
                     alpha=0.15, color='red', label='Chord lies above curve ✓')

axes[0].plot(1, 0, 'g*', markersize=20, zorder=10, label='Global minimum (only ONE!)')
axes[0].set_title('CONVEX Function (MSE Loss)\nOne minimum → GD always succeeds!', fontsize=12)
axes[0].set_xlabel('w (weight)'); axes[0].set_ylabel('Loss')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

# ── Right: Non-convex ──────────────────────────────────────────
axes[1].plot(w, L_nonconvex, 'steelblue', linewidth=3, label='Neural network loss (schematic)')

# Find local minima numerically
local_min_idx = argrelmin(L_nonconvex, order=15)[0]
global_min_idx = np.argmin(L_nonconvex)

for idx in local_min_idx:
    if idx != global_min_idx:
        axes[1].plot(w[idx], L_nonconvex[idx], 'rs', markersize=12,
                    zorder=10, label='Local minimum (TRAP!)')

axes[1].plot(w[global_min_idx], L_nonconvex[global_min_idx], 'g*',
             markersize=20, zorder=10, label='Global minimum')

# Show that chord goes BELOW the curve → non-convex
x1, x2 = -2.0, 1.5
y1, y2  = np.interp(x1, w, L_nonconvex), np.interp(x2, w, L_nonconvex)
axes[1].plot([x1, x2], [y1, y2], 'orange', linewidth=2.5, linestyle='--',
             label='Chord BELOW curve → non-convex ✗')

axes[1].set_title('NON-CONVEX Function (Neural Network)\nMultiple minima → GD may get trapped!', fontsize=12)
axes[1].set_xlabel('w (weight)'); axes[1].set_ylabel('Loss')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

plt.suptitle('Convex vs Non-Convex Loss Surfaces: House Price Model vs Neural Network',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Non-convex trap: starting point matters!
# Same learning rate, same function — different starting weights → different final solution
# ─────────────────────────────────────────────────────────────────

def L_nc(w):  return 0.5*w**4 - 2*w**2 + 0.3*w + 1
def dL_nc(w): return 2*w**3   - 4*w    + 0.3      # analytical derivative

def run_gd_1d(start, lr=0.05, n_steps=200):
    """Run gradient descent on the 1D non-convex function."""
    wt = start
    path = [wt]
    for _ in range(n_steps):
        wt = wt - lr * dL_nc(wt)       # gradient descent step
        wt = np.clip(wt, -3, 3)         # stay within plot range
        path.append(wt)
    return np.array(path)

starting_points = [-2.5, -1.5, 0.0, 1.0, 2.5]
colors = plt.cm.Set1(np.linspace(0, 0.8, len(starting_points)))

plt.figure(figsize=(12, 5))
plt.plot(w, L_nonconvex, 'k-', linewidth=2.5, zorder=5, label='Loss function')

for start, color in zip(starting_points, colors):
    path = run_gd_1d(start)
    # Plot start point
    plt.plot(start, L_nc(start), 'o', color=color, markersize=12, zorder=10)
    # Plot end point
    plt.plot(path[-1], L_nc(path[-1]), 's', color=color, markersize=10, zorder=10)
    # Draw the arrow from start to end
    plt.annotate('', xy=(path[-1], L_nc(path[-1])),
                 xytext=(start, L_nc(start)),
                 arrowprops=dict(arrowstyle='->', color=color, lw=2.5,
                                 connectionstyle='arc3,rad=0.3'))

plt.plot(w[global_min_idx], L_nonconvex[global_min_idx], 'g*', markersize=20,
         label='Global minimum', zorder=15)
plt.plot([], [], 'o', color='gray', markersize=10, label='Start (circle)')
plt.plot([], [], 's', color='gray', markersize=10, label='End (square)')

plt.title('Starting Point Determines Which Minimum GD Finds!\n'
          'Non-convex: different starts → different (possibly suboptimal) solutions', fontsize=12)
plt.xlabel('w (weight)'); plt.ylabel('Loss')
plt.legend(fontsize=9); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Practical takeaway for neural networks:")
print("  → Random initialization + multiple runs helps avoid bad local minima")
print("  → Good news: in high dimensions, most local minima have similar loss to global!")

## Part 2: Saddle Points — The Hidden Trap in Deep Learning

### Plain English First

A **saddle point** is like the center of a horse saddle or a mountain pass:  
- If you look along the horse's back: you're at the **lowest point** of a valley
- If you look sideways: you're at the **highest point** of a ridge

At a saddle point: `∇L = 0` (flat), but it's **not a minimum!**  
It's a minimum in some directions and a maximum in others.

**Why saddle points are tricky:**  
Gradient descent can get stuck at a saddle point because `∇L ≈ 0` — it thinks it found a minimum.  
In high-dimensional spaces (millions of weights), saddle points are **much more common** than local minima.

**Good news:** The noise from mini-batch SGD helps us escape saddle points!

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Visualize all types of critical points (where ∇L = 0)
# Three functions, three types of critical points at (0, 0)
# ─────────────────────────────────────────────────────────────────

x = np.linspace(-2, 2, 80)
y = np.linspace(-2, 2, 80)
X, Y = np.meshgrid(x, y)

# Three surface types
Z_bowl    =  X**2 + Y**2          # convex bowl → MINIMUM at (0,0)
Z_peak    = -(X**2 + Y**2)        # concave cap  → MAXIMUM at (0,0)
Z_saddle  =  X**2 - Y**2          # saddle       → SADDLE POINT at (0,0)

fig = plt.figure(figsize=(16, 5))

for idx, (Z, title, point_type) in enumerate([
    (Z_bowl,   'MINIMUM\nL = w₁² + w₂² (MSE loss bowl)\n∇L=0, all slopes upward from here',  'Minimum'),
    (Z_peak,   'MAXIMUM\nL = −w₁² − w₂²\n∇L=0, all slopes downward from here',               'Maximum'),
    (Z_saddle, 'SADDLE POINT\nL = w₁² − w₂²\n∇L=0 but NOT a minimum — min in one dir, max in other', 'Saddle'),
]):
    ax = fig.add_subplot(1, 3, idx+1, projection='3d')
    ax.plot_surface(X, Y, Z, cmap='viridis', alpha=0.85, rstride=3, cstride=3)
    ax.scatter([0], [0], [0], color='red', s=150, zorder=10)  # mark critical point
    ax.set_xlabel('w₁'); ax.set_ylabel('w₂'); ax.set_zlabel('Loss')
    ax.set_title(title, fontsize=10)

plt.suptitle('Three Types of Critical Points (all have ∇L = 0!)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────
# The Hessian test: how to identify the type of critical point
#
# The HESSIAN is the matrix of second derivatives:
#   H[i,j] = ∂²L / ∂wᵢ∂wⱼ
#
# At a critical point:
#   All eigenvalues of H > 0  →  LOCAL MINIMUM  (bowl)
#   All eigenvalues of H < 0  →  LOCAL MAXIMUM  (cap)
#   Mixed signs                →  SADDLE POINT
#
# Plain English: eigenvalues of H = "curvatures in different directions"
#   Positive curvature = concave up (like a valley floor)
#   Negative curvature = concave down (like a mountain top)
# ─────────────────────────────────────────────────────────────────

def classify_critical_point(H):
    """
    Given a Hessian matrix H, classify the critical point.
    H must be computed at a point where ∇L = 0.

    The eigenvalues of H are the curvatures in the principal directions.
    """
    eigenvalues = np.linalg.eigvals(H)    # compute eigenvalues of H
    if np.all(eigenvalues > 0):
        return f"LOCAL MINIMUM    (all eigenvalues > 0: {eigenvalues.round(2)})"
    elif np.all(eigenvalues < 0):
        return f"LOCAL MAXIMUM    (all eigenvalues < 0: {eigenvalues.round(2)})"
    else:
        return f"SADDLE POINT     (mixed signs: {eigenvalues.round(2)})"

# Hessians for our three surface types (computed analytically at (0,0)):
#   L = w1² + w2²    → H = [[2, 0], [0, 2]]
#   L = -w1² - w2²   → H = [[-2, 0], [0, -2]]
#   L = w1² - w2²    → H = [[2, 0], [0, -2]]

hessians = [
    ("L = w1² + w2²  (MSE bowl)",    np.array([[2.0, 0.0], [0.0,  2.0]])),
    ("L = -w1² - w2² (flipped bowl)", np.array([[-2.0, 0.0], [0.0, -2.0]])),
    ("L = w1² - w2²  (saddle)",       np.array([[2.0, 0.0], [0.0, -2.0]])),
    ("L with correlation (complex)",   np.array([[3.0, 1.0], [1.0,  2.0]])),
]

print("Hessian Analysis — Identifying Critical Point Types")
print("=" * 65)
for name, H in hessians:
    result = classify_critical_point(H)
    print(f"\n  {name}")
    print(f"  H = {H.tolist()}")
    print(f"  → {result}")

print()
print("Key practical note: Computing the full Hessian is O(n²) parameters")
print("For a 1M-parameter model: H would have 10¹² entries — impossible!")
print("In practice, we just watch whether the gradient stays near zero (saddle) or goes to zero (minimum).")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# SGD noise helps escape saddle points
#
# Near a saddle point, ∇L ≈ 0 → gradient descent stalls
# But SGD adds random noise → can kick the optimizer off the saddle
# ─────────────────────────────────────────────────────────────────

def loss_saddle(w):   return w[0]**2 - w[1]**2   # saddle at (0,0)
def grad_saddle(w):   return np.array([2*w[0], -2*w[1]])  # gradient

np.random.seed(42)

# Start very close to the saddle point (not exactly at it)
w_near_saddle = np.array([0.01, 0.01])   # slightly off-center

def run_with_noise(start, lr, n_steps, noise_std):
    """
    Gradient descent with optional noise.
    noise_std=0.0 → pure gradient descent
    noise_std>0   → SGD-like (noisy gradient)
    """
    w = np.array(start, dtype=float)
    path = [w.copy()]
    for _ in range(n_steps):
        g     = grad_saddle(w)                          # true gradient
        noise = noise_std * np.random.randn(2)          # random noise (simulates mini-batch noise)
        w     = w - lr * (g + noise)                    # update with possibly noisy gradient
        w     = np.clip(w, -3, 3)                       # keep in visible range
        path.append(w.copy())
    return np.array(path)

path_pure  = run_with_noise(w_near_saddle, lr=0.1, n_steps=100, noise_std=0.0)
path_noisy = run_with_noise(w_near_saddle, lr=0.1, n_steps=100, noise_std=0.05)

# Background: saddle surface contour
x_bg = np.linspace(-2, 2, 100)
y_bg = np.linspace(-2, 2, 100)
Xb, Yb = np.meshgrid(x_bg, y_bg)
Zb = Xb**2 - Yb**2

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
titles = ['Pure Gradient Descent\n→ gets stuck at saddle!',
          'Noisy GD (mini-batch SGD)\n→ escapes saddle!']

for ax, path, title in zip(axes, [path_pure, path_noisy], titles):
    cf = ax.contourf(Xb, Yb, Zb, levels=20, cmap='RdBu_r', alpha=0.7)
    plt.colorbar(cf, ax=ax, label='Loss')
    ax.contour(Xb, Yb, np.zeros_like(Zb), levels=[0], colors='black',
               linewidths=2, linestyles='--')  # the saddle "ridge"
    ax.plot(path[:,0], path[:,1], 'white', linewidth=1.5, alpha=0.9)
    ax.plot(path[0,0], path[0,1], 'gs', markersize=12, label='Start', zorder=10)
    ax.plot(path[-1,0], path[-1,1], 'r^', markersize=12,
            label=f'End ({path[-1,0]:.2f}, {path[-1,1]:.2f})', zorder=10)
    ax.plot(0, 0, 'k*', markersize=15, label='Saddle point (0,0)', zorder=10)
    ax.set_xlabel('w₁'); ax.set_ylabel('w₂')
    ax.set_title(f'L = w₁² − w₂²\n{title}', fontsize=11)
    ax.legend(fontsize=9)

plt.suptitle('Saddle Point Escape: Why SGD Noise is Actually Helpful!', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Pure GD: gradient is tiny near saddle → barely moves → stuck!")
print("Noisy SGD: random gradient noise kicks it off → finds the downhill path!")
print("This is one reason why mini-batch SGD outperforms full-batch GD in practice.")

## Part 3: The Good News About Deep Learning

You might think: "If neural networks are non-convex with many saddle points, how do they work?"

Modern research (Goodfellow, Choromanska, etc.) shows:

1. **Most local minima are good enough**: In high-dimensional spaces, local minima that exist tend to have similar loss to the global minimum
2. **Saddle points are more common than local minima** — but SGD noise helps escape them
3. **Overparameterized models** (more weights than training examples) often have exact solutions everywhere
4. **Empirically**: modern neural networks train reliably despite non-convexity

**The key insight**: In millions of dimensions, being stuck in a truly bad local minimum requires ALL dimensions to be trapped simultaneously — which is exponentially unlikely!

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Summary: Different types of critical points and how to detect them
# ─────────────────────────────────────────────────────────────────

print("SUMMARY: Types of Critical Points (where ∇L = 0)")
print("=" * 65)
print()
summary = [
    ("Global minimum", "∇L=0, H all positive eig", "The best possible weights",      "Goal!",            "Convex models guaranteed"),
    ("Local minimum",  "∇L=0, H all positive eig", "Good but maybe not best",          "Might trap GD",    "Deep networks"),
    ("Saddle point",   "∇L=0, H mixed eigenvalues","Min in some dirs, max in others",  "Slow GD (stalls)", "High-dim: very common"),
    ("Local maximum",  "∇L=0, H all negative eig", "The worst weights in a region",    "GD avoids (unstable)","Rare in practice"),
]
print(f"{'Type':15} | {'Hessian test':24} | {'Meaning':35} | {'GD behavior':20} | {'In practice'}")
print("-" * 120)
for row in summary:
    print(f"{row[0]:15} | {row[1]:24} | {row[2]:35} | {row[3]:20} | {row[4]}")

print()
print("For LINEAR REGRESSION with MSE loss: ALWAYS convex → GD always finds global minimum")
print("For NEURAL NETWORKS: non-convex → GD finds a local minimum (usually good in practice)")

---

## ✅ Self-Check Questions

**1.** The MSE loss of a linear regression model is convex. What does this guarantee about gradient descent?

**2.** You train a neural network. The loss stops decreasing after 500 steps. List two possible reasons:  
   (a) a "good" reason and (b) a "bad" reason.

**3.** At a saddle point, the gradient is zero. But how would you distinguish a saddle point  
   from a true minimum during training, without computing the full Hessian?

**4.** A function has Hessian `H = [[5, 2], [2, 5]]` at a critical point.  
   Is this a minimum, maximum, or saddle? (Hint: compute the eigenvalues.)

**5.** Why is mini-batch SGD often better than full-batch gradient descent for escaping  
   saddle points in neural networks?

<details>
<summary>Click to see answers</summary>

1. Gradient descent is **guaranteed to converge to the global minimum** — there is no local minimum to get trapped in.

2. (a) Good: it has converged to the optimal weights. (b) Bad: it is stuck at a saddle point or a local minimum with high loss.

3. Try **restarting from a different random initialization**. If gradient descent converges to a different (lower) loss value, the first stop was a saddle point or suboptimal local minimum. Also, if the gradient is very small but non-zero, you're near a saddle; if it's truly zero in all directions, you're at a minimum.

4. Eigenvalues of `[[5,2],[2,5]]`: the characteristic polynomial is `(5-λ)²-4=0` → `λ=3 or λ=7`. Both positive → **LOCAL MINIMUM**.

5. Mini-batch SGD uses a random subset of data each step, introducing **gradient noise**. Near saddle points where the true gradient is tiny, this noise can "kick" the optimizer in a productive direction, escaping the flat region.

</details>

---

## 📌 Summary

| Concept | One-line meaning | House price impact |
|---------|-----------------|-------------------|
| **Convex loss** | Bowl shape, one minimum | MSE linear regression: GD always succeeds |
| **Local minimum** | Best nearby, not globally | Neural network may find a "good enough" solution |
| **Saddle point** | Flat, but not a minimum | Can stall GD; SGD noise helps escape |
| **Hessian eigenvalues** | Curvatures of the loss surface | Classify critical points |
| **Deep learning reality** | High-dim losses are surprisingly well-behaved | Most local minima give good models |

**➡️ Next Notebook:** Probability Spaces — shifting gears from optimization to the probabilistic foundations of ML.